# AEM4L2 E08 - Avanzado: pipeline evaluado

**Objetivo:** combinar WER, resumen y confiabilidad en una decisión de pipeline.

**Uso en clase:** cierre o repaso final después de `e06_audio_pipeline_confiable.py`.


## Python exercise relacionado

- `python_puro/AEM4_python_exercises/AEM4L2_audio_pipelines/e06_audio_pipeline_confiable.py`


## Idea

Un pipeline confiable no resume siempre. Primero mide calidad. Si la transcripción no alcanza, no genera resumen automático.


In [ ]:
def simple_wer(reference: str, hypothesis: str) -> float:
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    errors = sum(1 for a, b in zip(ref, hyp) if a != b)
    errors += abs(len(ref) - len(hyp))
    return errors / len(ref) if ref else 0.0


In [ ]:
cases = [
    {
        "id": "support_clean",
        "reference": "el pedido cuatro cinco dos uno no llegó",
        "hypothesis": "el pedido cuatro cinco dos uno no llegó",
        "transcript": "El pedido 4521 no llegó y el cliente necesita una solución urgente.",
    },
    {
        "id": "support_noisy",
        "reference": "el pedido cuatro cinco dos uno no llegó",
        "hypothesis": "el perdido cuatro cinco no yo",
        "transcript": "Transcripción ruidosa con datos incompletos.",
    },
    {
        "id": "medical_critical",
        "reference": "tomar medicación cada ocho horas",
        "hypothesis": "tomar medicación cada dos horas",
        "transcript": "Indicación médica con posible error crítico.",
    },
]


## Gate simple

Regla inicial:

```text
WER <= 20% -> confiable
WER > 20%  -> revisión humana
```

Luego se puede mejorar agregando detección de errores críticos.


In [ ]:
WER_THRESHOLD = 0.20
CRITICAL_TERMS = {"ocho", "dos", "mil", "millón", "comprar", "vender"}

results = []
for case in cases:
    wer = simple_wer(case["reference"], case["hypothesis"])
    ref_terms = set(case["reference"].split())
    hyp_terms = set(case["hypothesis"].split())
    critical_error = bool((ref_terms ^ hyp_terms) & CRITICAL_TERMS)
    reliable = wer <= WER_THRESHOLD and not critical_error
    results.append({
        "id": case["id"],
        "wer": round(wer, 4),
        "critical_error": critical_error,
        "is_reliable": reliable,
        "decision": "summarize" if reliable else "human_review",
    })

results


## Lectura docente

El caso médico puede tener WER bajo y aun así bloquearse por error crítico. Esta es la diferencia entre métrica general y política de dominio.


In [ ]:
for result in results:
    print(f"{result['id']}: WER={result['wer']:.2%}, critical={result['critical_error']}, decision={result['decision']}")


## Desafío

Agregar thresholds por dominio:

```python
THRESHOLDS = {
    "ecommerce": 0.20,
    "medical": 0.05,
    "legal": 0.03,
    "informal": 0.40,
}
```

Luego decidir el gate usando `domain`.
